# Workplace Agent GRPO Training

This notebook trains a language model to act as a customer support triage agent using **Group Relative Policy Optimization (GRPO)**. The environment is a 3-step reinforcement learning loop — the agent must classify an incoming customer email, compose an appropriate reply, and decide whether to escalate to a human supervisor. The environment is deployed as a HuggingFace Space and grades each action using a rule-based reward policy that scores classification accuracy, reply quality (length, keywords, tone), and escalation correctness. GRPO generates multiple completions per prompt, scores them with the environment's reward function, and updates the model to favor higher-reward responses — no critic model needed.

In [ ]:
# Cell 2: Install dependencies
!pip install -q unsloth trl groq openenv-core wandb datasets pydantic
!pip install -q git+https://github.com/akshar-3011/meta-environment.git

In [ ]:
# Cell 3: Imports and configuration
import re
import os
import json
import sys
from typing import Any, Dict, List, Optional

import torch
from datasets import Dataset

# ── Configuration ──────────────────────────────────────────────────────
SPACE_URL = "https://akshar-3011-meta-environment.hf.space"
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
WANDB_PROJECT = "workplace-agent-grpo"

# Optional: set your W&B key here or via Colab secrets
# os.environ["WANDB_API_KEY"] = "your-key-here"

print(f"Space URL: {SPACE_URL}")
print(f"Model:     {MODEL_NAME}")
print(f"W&B:       {WANDB_PROJECT}")

In [ ]:
# Cell 4: Environment initialization and smoke test
# We use the local environment directly (installed from git)
sys.path.insert(0, ".")

from environment.workplace_environment import WorkplaceEnvironment
from core.graders import RuleBasedRewardPolicy
from models import WorkplaceAction
from data.scenario_repository import StaticScenarioRepository
from core.improvement.curriculum import CurriculumSampler

# Smoke test: create env, reset, print observation
policy = RuleBasedRewardPolicy()
env = WorkplaceEnvironment(reward_policy=policy)
obs = env.reset()

obs_dict = obs.model_dump() if hasattr(obs, "model_dump") else dict(obs)
print("✅ Environment connected successfully")
print(f"Email: {str(obs_dict.get('email', ''))[:100]}...")
print(f"Difficulty: {obs_dict.get('scenario_difficulty', 'unknown')}")
print(f"Category options: {obs_dict.get('category_options', [])}")

In [ ]:
# Cell 5: Reward function for GRPOTrainer

TAG_PATTERN = {
    "classify": re.compile(r"<classify>(.*?)</classify>", re.DOTALL),
    "reply":    re.compile(r"<reply>(.*?)</reply>", re.DOTALL),
    "escalate": re.compile(r"<escalate>(.*?)</escalate>", re.DOTALL),
}


def parse_agent_output(text: str) -> Dict[str, str]:
    """Extract classify/reply/escalate from XML-tagged completion."""
    result = {}
    for tag, pattern in TAG_PATTERN.items():
        match = pattern.search(text)
        result[tag] = match.group(1).strip() if match else ""
    return result


def openenv_reward_func(completions: list[str], **kwargs) -> list[float]:
    """Score each completion by running it through the environment.

    For each completion string:
      1. Parse <classify>, <reply>, <escalate> XML tags
      2. Instantiate a fresh environment and call reset()
      3. Call step() three times with parsed values
      4. Return cumulative reward (0.0 on any error)
    """
    results: list[float] = []

    for completion in completions:
        try:
            parsed = parse_agent_output(completion)

            policy = RuleBasedRewardPolicy()
            ep_env = WorkplaceEnvironment(reward_policy=policy)
            ep_env.reset()

            cumulative = 0.0
            for action_type in ["classify", "reply", "escalate"]:
                content = parsed.get(action_type, "")
                if not content:
                    content = "query" if action_type == "classify" else (
                        "no" if action_type == "escalate" else
                        "Thank you for reaching out."
                    )
                action = WorkplaceAction(action_type=action_type, content=content)
                obs = ep_env.step(action)
                obs_dict = obs.model_dump() if hasattr(obs, "model_dump") else dict(obs)
                cumulative += float(obs_dict.get("reward", 0.0) or 0.0)

            results.append(cumulative)

        except Exception as e:
            print(f"⚠️ Reward error: {e}")
            results.append(0.0)

    return results


# Quick test
test_completion = "<classify>refund</classify><reply>Hello, we will process your refund within 3-5 business days. We apologize for the inconvenience. Regards, Support Team.</reply><escalate>no</escalate>"
test_scores = openenv_reward_func([test_completion])
print(f"✅ Reward function test: score = {test_scores[0]:.4f}")

In [ ]:
# Cell 6: Model loading with Unsloth + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"✅ Model loaded: {MODEL_NAME} (4-bit + LoRA r=16)")
print(f"   Trainable params: {model.print_trainable_parameters()}")

In [ ]:
# Cell 7: Dataset construction from environment scenarios

SYSTEM_PROMPT = """You are a customer support triage agent. For each customer email, you must:
1. Classify the email into one of: refund, complaint, query
2. Write an appropriate reply
3. Decide whether to escalate to a human supervisor (yes/no)

Format your response using XML tags:
<classify>category</classify>
<reply>your reply text</reply>
<escalate>yes or no</escalate>"""


def build_prompt(email: str) -> str:
    """Build a chat-style prompt for the model."""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nCustomer email:\n{email}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


# Load easy scenarios from the corpus
sampler = CurriculumSampler()
easy_scenarios = [sc for sc in sampler.corpus if sc.get("difficulty") == "easy"][:25]

prompts = [build_prompt(sc["email"]) for sc in easy_scenarios]
dataset = Dataset.from_dict({"prompt": prompts})

print(f"✅ Dataset built: {len(dataset)} prompts from easy scenarios")
print(f"   Sample prompt preview:")
print(prompts[0][:200] + "...")

In [ ]:
# Cell 8: GRPOTrainer initialization
from trl import GRPOTrainer, GRPOConfig
import wandb

wandb.init(project=WANDB_PROJECT, name="grpo-workplace-run1")

grpo_config = GRPOConfig(
    output_dir="./grpo_output",
    max_prompt_length=512,
    max_completion_length=256,
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-6,
    logging_steps=1,
    save_steps=50,
    report_to="wandb",
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    config=grpo_config,
    train_dataset=dataset,
    reward_funcs=openenv_reward_func,
)

print("✅ GRPOTrainer initialized")
print(f"   Generations per prompt: {grpo_config.num_generations}")
print(f"   Epochs: {grpo_config.num_train_epochs}")
print(f"   Effective batch: {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")

In [ ]:
# Cell 9: Train and save
print("🚀 Starting GRPO training...")
trainer.train()

# Save the trained model
model.save_pretrained("grpo_workplace_agent")
tokenizer.save_pretrained("grpo_workplace_agent")

print("\n✅ Training complete!")
print("   Model saved to: ./grpo_workplace_agent")
print("   Check W&B dashboard for reward curves.")

wandb.finish()

## After Training

1. **Screenshot the W&B reward curve** — go to your W&B dashboard, find the `reward/mean` chart, screenshot it, save as `training_reward_curve.png`
2. **Commit to repo:**
   ```bash
   git add training_reward_curve.png grpo_workplace_agent/
   git commit -m "feat: GRPO training artifacts — reward curve + model checkpoint"
   git push origin main
   ```
3. **Add to README:** Reference the reward curve image and training results in your project README
4. **Optional — push model to HF Hub:**
   ```python
   model.push_to_hub("your-username/workplace-agent-grpo")
   tokenizer.push_to_hub("your-username/workplace-agent-grpo")
   ```